# Module 5 — GraphRAG

*Live session. Run the cells as you read.*

Four ways to answer a question about the graph of `Module 4`, and the question
is which one to reach for.

You open the box first. Then the four answer, then the same four fail, and then
there is a measurement of which works when — and of what that measurement is
not worth.

## Setup

The four pipelines need a served model and the graph module 4 built. Both are
named in `config.ini` — point it at your own model or at the one being served
for the class, and nothing else in the notebook changes.

Install once, from the repository root, not from a cell:

```
pip install -e '.[graphrag,notebooks]'
```

Every version is exact there, down to the patch. Installing from inside a
notebook puts the pins in two places, and a version written twice diverges.

In [ ]:
import configparser
import json
from pathlib import Path

from IPython.display import SVG, Markdown, display
from neo4j import GraphDatabase

config = configparser.ConfigParser()
if not config.read(Path("config.ini")):
    raise SystemExit("config.ini not found — copy config.ini.example and fill it in")

driver = GraphDatabase.driver(
    config["neo4j"]["uri"],
    auth=(config["neo4j"]["user"], config["neo4j"]["password"]))

# One session, open for the whole notebook. Every pipeline and every tool below
# takes it as its first argument: that is the one calling convention here.
session = driver.session(database=config["neo4j"]["database"])


def run(query, **params):
    """Run one Cypher query and return its rows as a list of dictionaries."""
    return [dict(record) for record in session.run(query, **params)]


# The two helpers every drawing needs, same as module 4 and for the same two
# reasons: an unescaped ampersand stops the whole image, and a label longer than
# its box draws over whatever is beside it.
def xml(text):
    """What SVG cannot swallow. One vendor is called `A & M AMUSEMENT INC`."""
    return str(text).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def fit(text, width, size):
    """Shorten a label to what its box holds, at roughly 0.55 em per character."""
    room = int((width - 16) / (size * 0.55))
    return text if len(text) <= room else text[:room - 1] + "\u2026"


def show_json(rows, indent=2):
    """Display rows as JSON instead of a one-line Python repr.

    `default=str` is not cosmetic: Neo4j returns its own temporal types, and
    `json.dumps` raises TypeError on any result carrying a date.
    """
    display(Markdown("```json\n"
                     + json.dumps(rows, indent=indent, ensure_ascii=False, default=str)
                     + "\n```"))


from pipelines import graphrag, rag, text2cypher
from pipelines.react import agent

print("graph:", config["neo4j"]["database"], "| model:", config["llm"]["model"])
print("nodes:", run("MATCH (n) RETURN count(n) AS n")[0]["n"])

## Opening the box

Three of the four systems you are about to meet are built with LangGraph, and
the whole library is two ideas. Here they are with nothing else in the frame:
no graph, no documents, no model.

### Idea one: the state is a dict, and nodes return a piece of it

In [ ]:
from operator import add
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import END, START, StateGraph

class Toy(TypedDict):
    trail: Annotated[list[str], add]   # <- the reducer: append, do not replace

def first(state):  return {"trail": ["first"]}
def second(state): return {"trail": ["second"]}

toy = StateGraph(Toy)
toy.add_node(first)
toy.add_node(second)
toy.add_edge(START, "first")
toy.add_edge("first", "second")
toy.add_edge("second", END)

WITH = toy.compile().invoke({"trail": []})
WITH

In [ ]:
from pictures import checks, toy_picture

picture = toy_picture(WITH["trail"], reducer=True)
print(checks(picture))
display(SVG(picture))

No node modified anything: each returned `{"trail": [...]}` and LangGraph
merged it.

**Now remove the reducer** and run the identical graph.

In [ ]:
class ToyWithoutReducer(TypedDict):
    trail: list[str]        # no Annotated, no add

plain = StateGraph(ToyWithoutReducer)
plain.add_node(first)
plain.add_node(second)
plain.add_edge(START, "first")
plain.add_edge("first", "second")
plain.add_edge("second", END)

WITHOUT = plain.compile().invoke({"trail": []})
WITHOUT

In [ ]:
picture = toy_picture(WITHOUT["trail"], reducer=False)
print(checks(picture))
display(SVG(picture))

**`first` is gone.** Compare the third box with the one in the picture above:
the second node's value **replaced** the first instead of being appended, and
nothing raised, nothing warned. You would keep this result and believe it.

One line of type annotation is the whole difference between the two graphs.

Hold on to that shape — it is the shape of every other failure in this module.
Nothing breaks. You just get a smaller true-looking answer.

### Idea two: an edge can be chosen at run time

In [ ]:
class Counter(TypedDict):
    n: int

def bump(state):
    return {"n": state["n"] + 1}

def again(state):
    # Returns the NAME of the next node. This is the whole of "an agent loops".
    return "bump" if state["n"] < 3 else END

loop = StateGraph(Counter)
loop.add_node(bump)
loop.add_edge(START, "bump")
loop.add_conditional_edges("bump", again)

app = loop.compile()
PASSES = list(app.stream({"n": 0}))   # one entry per turn of the loop
FINAL = app.invoke({"n": 0})
PASSES, FINAL

A cycle is `add_conditional_edges` and nothing else. Here are the same three
lines in `pipelines/text2cypher.py`:

```python
graph.add_edge("generate", "validate")
graph.add_conditional_edges("validate", after_validation)   # errors left? correct. else execute.
graph.add_edge("correct", "validate")
```

`after_validation` looks at whether the validator returned errors. **The result
of the previous node chooses the edge.**

In [ ]:
from pictures import loop_picture

picture = loop_picture(PASSES, FINAL)
print(checks(picture))
display(SVG(picture))

One box, because there is one node. The row underneath is the same node three
times, and the dashed edge is the one `again()` chose each time.

## The four systems, working

Four ways to answer a question about this graph. Watch them run before asking
whether they are any good: each returns the answer **and what it read**, because
an answer whose sources you cannot see is indistinguishable from one the model
made up.

The fourth is the one that chooses, so it arrives with its tools laid out first.

In [ ]:
from pictures import systems_picture
from pipelines.graphrag import build as graphrag_graph
from pipelines.rag import build as rag_graph
from pipelines.react import build as react_graph
from pipelines.text2cypher import build as text2cypher_graph

# Nothing below is typed by hand: the nodes and the edges are read out of the
# four compiled graphs, so a node renamed in `pipelines/` renames itself here.
# All four are LangGraph graphs, the agent included — which is the point of the
# picture, because the difference between them is a shape and not a library.
SHAPES = {
    "RAG": rag_graph().get_graph(),
    "GraphRAG": graphrag_graph().get_graph(),
    "Text2Cypher": text2cypher_graph().get_graph(),
    "Agent": react_graph().get_graph(),
}

picture = systems_picture(SHAPES)
print(checks(picture))
display(SVG(picture))

Three fixed routes and one cycle.

A dashed edge is one LangGraph calls **conditional** — the next node is chosen
while the thing runs, from what the previous one returned. RAG and GraphRAG have
none: their route is the same for every question ever asked. Text2Cypher has
four, and three of them are the repair loop. The agent is two nodes and nothing
else, and the edge between them is chosen by the model.

That is the whole taxonomy. The rest of this section is the four of them
answering.

### RAG — read the pages, answer from them

In [ ]:
# The question somebody renewing this contract asks. "Liquidated damages" is the trade's
# name for a penalty fixed in the contract rather than argued afterwards.
TECHNICAL = "What is the liquidated damages amount per day?"

rag(session, TECHNICAL, k=3, dense=False).show(chars=400)

Three things came back: the answer, the pages it came from, and **the exact text
the model was shown**. The last one is what to look at when an answer surprises
you.

And it read the city's standard terms, not one contract. `liquidated damages`
appears on 3,345 pages across **1,033 contracts**; the three retrieved are the
same form at three revisions. For a question this general that is the right
answer — the clause really is everybody's.

### The same retriever, the other index

In [ ]:
# `dense=True` embeds the question and matches meaning instead of words — and
# the word this question uses appears on none of the pages it finds.
CANCEL = "Can the city cancel the contract?"

rag(session, CANCEL, k=3, dense=True).show(chars=400)

Right, and quoted from the clause itself: *"the City may terminate this Contract
for cause or convenience."*

**The word `cancel` is on none of the three pages.** They say `terminate` — once,
eleven times, sixteen times. The question and the document share no word, and the
dense retriever found it anyway. That is what embedding a question buys
over matching its words.

In [ ]:
from pictures import checks, rag_picture
from pipelines.retrievers import retrieve

CANCEL_ANSWER = rag(session, CANCEL, k=3, dense=True)
CANCEL_PAGES = retrieve(session, CANCEL, k=3, dense=True)

picture = rag_picture(CANCEL_ANSWER, CANCEL_PAGES, dense=True)
print(checks(picture))
display(SVG(picture))

**The same picture as the one above, with one box changed.** `page_embedding`
instead of `page_text`, and everything downstream is identical: three pages, the
same cut at 1,500 characters, one context.

The pipeline does not know which index it read from. Swapping the retriever is a
boolean, and that is the whole of the difference between the two cells — which
is worth seeing before the next section, where two systems differ by a node and
not by a flag.

In [ ]:
from pictures import rag_picture
from pipelines.retrievers import retrieve

RAG_ANSWER = rag(session, TECHNICAL, k=3, dense=False)
RAG_PAGES = retrieve(session, TECHNICAL, k=3, dense=False)

picture = rag_picture(RAG_ANSWER, RAG_PAGES)
print(checks(picture))
display(SVG(picture))

The orange is the part of each page the context does not carry. The node that
assembles it keeps the first 1,500 characters of a page, and on this question
the pages run to over 3,000 — so of the eight dollar amounts on the top page,
the model saw six.

Nothing downstream can tell. The answer names the amounts it was shown and has
no way to mention the ones it was not.

### GraphRAG — the same pages, with what the graph knows about them

One node different from RAG: `attach_neighbours` instead of `assemble_context`.
And a question with an **object** this time — *who is the vendor* on its own has
none, so the answer would be about whichever contract retrieval happened to
surface.

In [ ]:
# A question with an object. "Who is the vendor" on its own has none, and the
# answer would be about whichever contract retrieval happened to surface.
SUPPLY = "Who supplies intraosseous systems to the Fire Department?"

graphrag(session, SUPPLY, k=2, dense=False).show(chars=400)

Read the context, not the answer. Every block now opens with a line that came
from no page:

```
graph: vendors: TELEFLEX LLC; departments: CHICAGO FIRE DEPARTMENT; revisions: 2
```

The model even says where it read it. Check whether it had any choice:

In [ ]:
from pipelines.retrievers import neighbours, retrieve

# The two halves, side by side: what the page says, and what the graph adds.
PAGES = retrieve(session, SUPPLY, k=2, dense=False)
FACTS = {p["purchase_order"]: neighbours(session, p["purchase_order"]) for p in PAGES}

for page in PAGES:
    graph = FACTS[page["purchase_order"]]
    shown = " ".join(page["text"].split())[:1500].upper()
    vendor, department = graph["vendors"][0], graph["departments"][0]
    print(f"PO {page['purchase_order']} page {page['page']}")
    print(f"   vendor      {vendor:34} on the page? "
          f"{'yes' if vendor.upper()[:14] in shown else 'NO'}")
    print(f"   department  {department:34} on the page? "
          f"{'yes' if department.upper()[:16] in shown else 'NO'}")
    print(f"   revisions   {graph['revisions']:<34} "
          f"no single page can say this")

**The vendor is on neither page.** The pages say what the city is buying — *"1228910A:
Intraosseous Vascular Access System, Accessories and Supplies"* — and never who
sells it. One page names the department, the other names neither. Without that
line the question has no answer in the text at all.

And `revisions: 2` is a fact no single page can carry, because it is a count over
pages' own ancestors.

In [ ]:
from pictures import graphrag_picture

picture = graphrag_picture(PAGES, FACTS)
print(checks(picture))
display(SVG(picture))

### Text2Cypher — the graph answers, and the model writes the query

The other thing called GraphRAG, and the opposite of the one above: there the
traversal was fixed and written by hand, here the model writes it. So this is the
only approach that gets the **whole measured schema** — and watch what it does
with it.

`text2cypher_with_trace` is `text2cypher` with every draft printed: the query as
it stood, and who objected to it. Two questions, and the first one works.

In [ ]:
from pipelines.text2cypher import text2cypher_with_trace

# Not a plain count: a traversal, a grouping, a filter on the aggregate and a
# count of what survives. The question here was "which vendor holds the most
# contracts?" until 5/11/2026, when it started answering with a syntax error
# three runs out of three -- it had been the marginal case for days, and the
# battery caught it drifting from 2 of 3 to 0 of 3 in one morning.
BUSY = "How many departments award more than 1000 contracts?"

text2cypher_with_trace(BUSY);

**Twenty-one departments**, and it took a traversal, a grouping, a filter on the
aggregate and a count of what survived — written by the model, from a schema it
was handed and nothing else.

And `nothing objected to it`: one draft, no correction. That line is new. Until
5/11/2026 the served model objected to this query three times out of three, each
objection quoting the schema line that refuted it, and the loop spent its whole
budget arriving back where it started. The change was not the prompt — three
rewordings of it made things worse — it was the model. `Where each one breaks`
has the loop with something real to do.

### Agent — the tools first

The other three systems have one route and no choices. This one has eight tools
and picks among them, so the tools come before the run: what it can reach is
what the shape cannot tell you.

In [ ]:
from pictures import tools_picture
from pipelines.react import tools_for

picture = tools_picture(tools_for(session))
print(checks(picture))
display(SVG(picture))

Two read text, three read the graph, one rewrites a question, one ends the run.
Nothing else is in front of the model — no schema, no list of labels, no
examples.

And the grouping is the lesson. `search_descriptions` and `search_titles` can
find a contract by subject and cannot say who won it; `find_entity`,
`contracts_of` and `about_entity` can say who won it and cannot find a subject.
Neither half answers a question that needs both, which is why there are eight
tools and not one.

`reformulate_query` is the odd one: it touches no data at all. It turns a
question into the words a clerk would have typed, and it exists because of the
failure in `Where each one breaks`.

### And now it runs

This takes longer than the others, because it is several round trips instead of
one.

In [ ]:
PO_RUN = agent(session, "Which departments work with the vendor that holds "
                           "purchase order 82547 today?")

Read the steps, not the answer.

The first step is **wrong**: `find_entity` resolves names, and `82547` is not a
name. The second recovers with a different tool. That is the loop doing its job,
and it is the only thing here that can do it.

The question needs the vendor **before** the question about departments can be
asked. What an agent buys is not a better retriever — it is somewhere to keep an
intermediate result.

In [ ]:
from pictures import react_picture

picture = react_picture(PO_RUN["trace"], PO_RUN["sent"])
print(checks(picture))
display(SVG(picture))

### The agent again, on a question with two conditions

In [ ]:
# Two conditions on the same contracts: a vendor and a department. A different
# route, not just a longer one.
BOTH = ("Which contracts does Erie Neighborhood House have with the "
        "Department of Public Health?")

BOTH_RUN = agent(session, BOTH)

A different route, and not just a longer one: a `find_entity` per condition,
then **one** `contracts_of` carrying `within`. Twenty-two contracts, and no list
was compared against another — the intersection is a query.

How many steps that takes moves between runs. Measured on 4/11/2026: once it
resolved the two names in two steps, once it asked for both in the same step and
the loop was a step shorter. The tools it reaches for do not move; the number of
round trips does.

That is what separates this from the three above. Their route is the same for
every question; this one's is chosen a step at a time, and a question of a
different shape gets a different route.

Two details in the trace are worth stopping on.

The model wrote `'id': 'L3430'` without the `vendor:` prefix, and it worked. The
tool completed an identifier it had issued one step earlier; it would have
refused one the model invented.

And the cost: **39,233 characters put in front of the model, against 15,402** for
the question above. One more condition, two and a half times the context. An agent
pays for every turn twice: once to ask, and once more because the whole
conversation goes back up with the next question.

In [ ]:
picture = react_picture(BOTH_RUN["trace"], BOTH_RUN["sent"])
print(checks(picture))
display(SVG(picture))

### The agent again, when your words are not the city's

`Where each one breaks` has a question that defeats every retriever: the words
you have are not the words the document uses. The agent has a tool for exactly
that — and the question is whether it reaches for it.

In [ ]:
# You say "cleaning". The city writes "janitorial".
agent(session, "Which contracts are for cleaning city buildings?");

**Step 2 is the one to read.** `search_descriptions` came back with nothing — and
with a line telling the agent what to do about it: *"these descriptions cover
only 130 of 88,916 contracts: call `reformulate_query` and then
`search_titles`."*

It did. And look at what `reformulate_query` returned:

> `"CLEANING" OR "CLEANING SERVICES" OR "BUILDING CLEANING" OR "FACILITY
> CLEANING" OR "MAINTENANCE AND CLEANING" OR "JANITORIAL SERVICES" OR
> "JANITORIAL" …`

**`JANITORIAL` is in there, and nothing in `pipelines/search.py` put it there.**
The model already knew the trade's word for cleaning; it had simply never been
asked. The tool does not search anything — it turns your question into the words
a clerk would have typed, and hands them to the search that does.

Four steps, and the contracts it returns are all titled *Janitorial Services*.

The retrievers of the first two systems cannot do this. Not because they are
worse at retrieval, but because **they have no step at which to change the
question.** That step is what the fourth system is.

### The four, side by side

Now that all four have run, the difference between them is a shape:

| | shape | who decides the order | what the graph does |
|---|---|---|---|
| RAG | `retrieve -> assemble -> answer` | we do, before the question exists | nothing |
| GraphRAG | `retrieve -> attach_neighbours -> answer` | we do | enriches the context |
| Text2Cypher | `generate -> validate ⇄ correct -> execute` | we do, with a bounded loop | answers |
| agent | `model -> tool -> model -> ...` | **the model, one step at a time** | answers, or searches |

**All four are graphs**, and what singles the agent out is not the absence of a
framework. Look at the conditional edges. Text2Cypher has three, and all three
are decided by what the previous node *returned* — errors left, or none. The
agent has one, and it is decided by whether the **model asked for a tool**.

A framework is not what makes a loop agentic. The model choosing the next edge
is.

**Two of these are called GraphRAG**, and that is not sloppiness: the term names
a family — approaches where the graph takes part — and in the wild it means
either one. Tell them apart by what the graph does: enrich the context, or
answer. A student who leaves knowing the word means two things, and that you
have to ask which, has learned something that pays off in the first meeting they
walk into.

Open `pipelines/rag.py` and `pipelines/graphrag.py` side by side. They differ by
**one line of graph** and one node. They are not two systems, they are two
wirings — and that is worth knowing before anyone argues about which is better.

## Where each one breaks

The same four, in the same order, and each one failing. Every cell here has a
twin in `The four systems, working`: same system, same shape of call, and this time the answer is
wrong — or looks wrong in a way worth naming.

### RAG — the same question, in the words you actually have

`The four systems, working` asked for the *liquidated damages amount per day* and got it. Here is
the same question, from somebody who has never read one of these contracts.

In [ ]:
# The same question, in the words of somebody who has never read one of these
# contracts -- which is everybody, the first time.
PLAIN = "What happens if the contractor finishes late?"

rag(session, PLAIN, k=3, dense=False).show(chars=400)

Nothing. The top page is a retiree benefit booklet about prescription drugs.

It is the same question, and the answer is still on 1,033 contracts. What changed
is that `liquidated damages` is the trade's phrase and `finishes late` is not:
retrieval matches the words **the document** uses, not the ones you have. The
dense retriever does no better here — swap `dense=True` and see. Meaning-search
crosses a synonym; `finishes late` reaching `liquidated damages` is not a
synonym but an inference about how procurement works, and no embedding carries
it.

Worth saying out loud: **the model refused**. It had pages in front of it and
could have assembled something; instead it said the context does not contain the
answer. That is the one reason this failure is visible at all.

### GraphRAG — the question one hop cannot answer

In [ ]:
# A question whose answer is spread over two contracts, not one.
SUPPLIERS = "Which suppliers has the city used for intraosseous vascular access systems?"

SUPPLIERS_ANSWER = graphrag(session, SUPPLIERS, k=6, dense=False)
print(SUPPLIERS_ANSWER.answer[:300], "\n")
# Six pages, one purchase order -- so one hop, not six. The blocks carry the
# revision because each filing of an order has its own page 2.
print("what it retrieved:")
for line in SUPPLIERS_ANSWER.context.splitlines():
    if line.startswith("["):
        print("   " + line)
print("\nthe graph lines it received, one per page:")
for line in SUPPLIERS_ANSWER.context.splitlines():
    if line.startswith("graph:"):
        print("   " + line[:92])

# What the graph holds on the other contract for the same thing.
print("\nand what the city actually bought, both contracts:")
show_json(run("""
MATCH (c:Contract)-[v:HAS_VENDOR]->(o:Organization)
WHERE c.id IN ['214209', '82547']
RETURN c.id AS po, o.name AS vendor,
       toString(v.validFrom) AS from, toString(v.validTo) AS to
ORDER BY po, from
"""))

**`ARROW INTERNATIONAL` is missing, and it is an answer.** The city bought
intraosseous systems from Arrow from 2018 to 2021, on purchase order 82547, and
from Teleflex since. Six pages came back and every one of them belongs to a single
contract, so every `graph:` line says the same thing. Six copies of one hop is
still one hop.

Look at the traversal to see why nothing could have gone differently:

```cypher
MATCH (c:Contract {id: $purchase_order})
OPTIONAL MATCH (c)-[:HAS_VENDOR]->(o:Organization)
OPTIONAL MATCH (c)<-[:ASSIGNS_CONTRACT]-(d:Department)
OPTIONAL MATCH (c)<-[:INCLUDED_IN_CONTRACT]-(r:ContractRecord)
```

**One hop, from the contract the page belongs to.** There is no step from a
vendor to that vendor's *other* contracts, so nothing here can join page A to
page B. This GraphRAG annotates; it does not follow.

That second hop exists in this module — the vendor's reach, every contract it
holds — but it is a **tool the agent can call**, not a node in a fixed pipeline.
A pipeline decides its route before the question is asked, and a question whose
answer is spread over two contracts needs a route chosen after the first result
comes back.

Which is the fourth system, and the section after this one.

### Text2Cypher — the same pipeline, the wrong answer

`The four systems, working` counted departments and the answer was right. Three
drafts here too, and the answer is **0**.

In [ ]:
FIRE = "How many contracts is the Chicago Fire Department assigned?"

FIRE_RUN = text2cypher_with_trace(FIRE)

**The answer is 0. The Chicago Fire Department holds 1,470.**

Read the drafts, because nothing here went the way it looks.

**Draft 1** filters `Department` on `name`. The objection is *correct and
useful*: `Department` has no `name`, the name lives on `DepartmentGroup`, and
the way across is `BELONGS_TO_ORG_GROUP`. All of that is in the schema and all
of it is true.

**Draft 2** follows that advice exactly, and is a good query.

**Draft 3** is draft 2, unchanged — because there was nothing left to fix.

And the query that ran is right in every way the schema can describe. The one
thing between it and the answer is **letter case**:

```cypher
WHERE dg.name CONTAINS 'Chicago Fire Department'   //     0
WHERE dg.name CONTAINS 'CHICAGO FIRE DEPARTMENT'   // 1,470
```

The schema carries property *names*, not property *values*. Nothing in this loop
could have caught it: `EXPLAIN` reads syntax and accepted all three drafts, and
the model reads a schema that does not contain the fact it needed.

**One more thing, in the objections themselves.** Look at how the one on drafts
2 and 3 ends: *"this is a semantic risk, not a schema violation, so I will not
flag it as a defect."* It said it would not flag it, and then returned it in the
list of defects — which is how two corrections changed nothing. The prompt in
`pipelines/prompt.py` asks for an empty list when the query is correct, in those
words, and this is what asking gets you.

**And the two Text2Cypher cells are indistinguishable from outside.** Same
pipeline, same three drafts, same exhausted budget, no error either time. One
answer is right and one is wrong, and nothing in the path says which.

`validTo` is not in the source data. It is the `validFrom` of the next row,
written by a derivation of ours that assumes two vendor numbers mean two
companies. The city reissues that number when a company renames itself.

So the graph states a change no source supports — **with dates on it**, and a
fact with a date reads as observed rather than inferred. Nobody reading it can tell that from
the graph alone, and that is what makes it worth finding.

### Agent — the question its tools cannot ask

Same agent as `The four systems, working`, same eight tools. One question from
there, put to it again — the one Text2Cypher answered in a single draft.

In [ ]:
# Text2Cypher answered this with one query: 21.
COUNT_RUN = agent(session, "How many departments award more than 1000 contracts?")

**Twenty-one is the answer, and it has not reached it once.**

Read the steps. `find_entity` finds an entity **by name**, so the agent is
guessing department names one at a time — and every guess comes back with a
real count, which is what makes it keep going. Five runs on 5/11/2026: three
said 11, one said *"eight"* and then listed nine of them in the same sentence,
and one spent the whole twelve-step budget and stopped with no answer at all.

What it says varies. That it is wrong does not.

And in one of those runs it said why, correctly: *"the full city roster of
departments is not enumerated by the tools."* A question about **all**
departments needs their names before it starts, and nothing the agent can call
will give them.

That is the shape of this failure, and it is not a failure of reasoning. The
agent beat the other three on four of the five questions in `The measurement`,
by choosing its route. Here there is no route to choose, and choosing cannot
invent one — while the system with the narrowest shape of all, one query against
the whole schema, answered the same question in a single draft.

**An agent costs more when it fails than when it succeeds**, and the run above
says how much.

No error either time. One returns **nothing**, the other returns **a plausible
row** saying one department holds 86,534 contracts, with `department: null`.

Measured on 1/9/2026 the agent hit both and drew opposite wrong conclusions from
the same cause: given zero rows it announced the purchase order is absent; given
the null row it took it as an answer and built two more steps on top.

Module 4 teaches that a `MATCH` on a missing relationship returns zero rows and
no error. This is one step further: a missing **property** returns a row.

## The measurement, and what it is worth

*Is GraphRAG Needed?* (AWS/Cisco, GEM 2026) asks **which approach works
when**. This is that question put to these four, on this graph, with this
model.

Five questions, chosen because they **separate** the four — not because they are
hard. One band per row: a count only the graph holds, a fact all four can reach,
one that needs text and graph together, one whose words are not the document's
words, and one that names two things at once.

Four of the five are questions you have just watched run. The fifth —
*who is the supplier of purchase order 82547 today* — is the floor: the one all
four answer, on the order the agent section uses twice. Without a row everyone
passes, the rows they fail do not mean anything.

Three runs of each, 5/11/2026.

In [ ]:
from pipelines.compare import GRID, RECORDED, SECONDS, table

print(table(RECORDED, SECONDS))

**No system covers the grid, and the agent covers the most.** That is the
paper's finding, reproduced here on contracts it has never seen.

Read down the columns and each has a shape. RAG and GraphRAG answer what a page
says. Text2Cypher answers what the graph counts, and nothing a page says.
The agent answers four bands of five — and fails the one the graph counts.

Two marks are worth more than the table shows.

**The agent says 11 where the answer is 21** — and it says why: *"I can only
count the departments that appear in the graph."* It counted the departments its
tools had returned, not the departments there are. A wrong answer, with its own
reason attached.

**GraphRAG is right on the two-part question for a narrower reason than the
question asked.** It names the department *of that purchase order*; the question
asked which departments work with *the vendor*. The two coincide because this
vendor has one. The agent answers what was asked — *"it works with only one
department"* — and the two answers are the same string for different reasons.

### Does the agent do better if it can see the schema?

It is the obvious thing to try, and the one scenario the measurement above does
not cover: the agent is the only system here that never sees the shape of the
graph. `agent(..., schema=True)` appends the whole measured schema —
seventeen thousand characters — to its instructions.

Five questions, three runs each.

In [ ]:
from pipelines.compare import schema_table

print(schema_table())

**It never did better, on any of the five.** Three to seven times the context,
two to five times the time, and the same answers — except where they got worse.

Two rows carry the result.

On **the count**, the answer became *unstable*: 8, then 10, then 8. Without the
schema it was the same wrong number — 11 — three times running. Wrong and
reproducible became wrong and not.

On **the vocabulary question** it spent the entire twelve-step budget, all three
runs, and returned **nothing at all**. Without the schema: seven steps and an
honest refusal. The `yes` in that row is the measurement being literal — three
empty answers do agree with each other.

And the reason is not mysterious. The agent's tools do not let it write Cypher,
so the schema is context it has **no way to act on**. All it can change is which
tool the model reaches for, and it changed that for the worse: more to read,
more steps, and on one question it never stopped reaching.

**More true context is not free.** That is a production lesson, not a teaching
one, and it is the opposite of what the obvious thing to try predicts.

In [ ]:
# Nothing in this notebook wrote to the graph. Everything above is a read.
session.close()
driver.close()
print("done")